# DROID — the 3D and 2D track ground truth, every episode

One job: look at DROID's **3D tracks** and the **2D tracks they project to in each view**, for
**all 50 episodes**, and judge by eye whether they are right.

It reads the dataset already on disk at `~/tapvidmv_workspace/tapvidmv_dataset/droid` — nothing is
downloaded, and no repository module is imported.

**The one contract being checked.** `tracks_xyz.npy` `(T, N, 3)` is the 3D ground truth, in world
coordinates, shared by all three cameras. The 2D ground truth is not stored separately: it is that
3D track pushed through a view's own per-frame `extrinsics_w2c` and its `(fx, fy, cx, cy)`, with
`visibility.npy` `(T, N)` saying whether that view can actually see it. So if a dot sits on the
right physical spot in all three cameras at once, and its 3D curve sits on the scene surface, the
ground truth is right — and every panel below is built to make that judgement easy.

**Reading the panels**

| | |
| --- | --- |
| filled dot | the view marks this point visible on this frame |
| hollow dot | occluded, or projected outside the image |
| trail | *past world positions* seen through the *current* camera, so a world-static point leaves **no** trail even in the moving wrist view |
| colour | one colour per track, **identical in 2D and in 3D**, so a dot can be matched to its 3D curve |

The rig: view 0 is the wrist camera riding on the arm, views 1 and 2 are fixed exteriors.

**Sections 5 and 6 sweep all 50 episodes** — 5 does the 2D tracks, 6 does the 3D tracks.


In [ ]:
#@title Setup
import functools
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import mediapy
import numpy as np

plt.rcParams["figure.dpi"] = 110
print("ready")


## 1. The reader, the projection, and the drawing

Three cells, run once. Depth is memory-mapped and images stay compressed until a frame is asked
for, so opening an episode is instant and the 50-episode sweeps stay in RAM.


In [ ]:
#@title The reader — local files only (run once)
# The packaged export, already on disk. Point this anywhere else if you move it.
DATA_ROOT = Path("~/tapvidmv_workspace/tapvidmv_dataset/droid").expanduser()


def find_episodes(root=None):
    """Every episode directory that actually holds tracks."""
    root = DATA_ROOT if root is None else root
    assert root.is_dir(), f"no DROID dataset at {root}"
    return sorted(p.name for p in root.iterdir()
                  if p.is_dir() and (p / "tracks_xyz.npy").exists())


def decode_jpeg(raw):
    """Decode one frame's JPEG bytes to an RGB uint8 image."""
    buffer = np.frombuffer(bytes(raw), dtype=np.uint8)
    return cv2.cvtColor(cv2.imdecode(buffer, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def project_tracks(tracks_xyz, intrinsics, extrinsics_w2c):
    """Project world points through one view's cameras; returns (xy, z).

    This is the whole contract between the 3D ground truth and the 2D ground
    truth: ``tracks_xyz`` is in world space, and a view's per-frame
    ``extrinsics_w2c`` plus its ``(fx, fy, cx, cy)`` put it on that view's
    pixels, under the integer-centre convention.
    """
    points_h = np.concatenate(
        [tracks_xyz, np.ones((*tracks_xyz.shape[:2], 1), dtype=np.float32)], axis=-1)
    points_camera = np.einsum("tij,tnj->tni", extrinsics_w2c, points_h)
    z = points_camera[..., 2]
    with np.errstate(invalid="ignore", divide="ignore"):
        xy = points_camera[..., :2] / z[..., None]
    xy = xy * intrinsics[None, None, :2] + intrinsics[None, None, 2:]
    return xy.astype(np.float32), z.astype(np.float32)


def camera_centers(extrinsics_w2c):
    """World-space camera centres from world-to-camera transforms."""
    rotation, translation = extrinsics_w2c[:, :3, :3], extrinsics_w2c[:, :3, 3]
    return -np.einsum("tji,tj->ti", rotation, translation)


@dataclass(eq=False)
class View:
    index: int
    intrinsics: np.ndarray        # (4,) fx, fy, cx, cy
    extrinsics_w2c: np.ndarray    # (T, 4, 4)
    visibility: np.ndarray        # (T, N)
    jpegs: np.ndarray             # (T,) object array of JPEG bytes
    path: Path

    def image(self, frame):
        return decode_jpeg(self.jpegs[int(frame)])

    def depth(self, frame):
        """Read one depth frame off disk, or None when this view has none."""
        path = self.path / "depth.npy"
        if not path.exists():
            return None
        return np.asarray(np.load(path, mmap_mode="r")[int(frame)], dtype=np.float32)

    @functools.cached_property
    def centers(self):
        return camera_centers(self.extrinsics_w2c)

    @functools.cached_property
    def camera_motion_m(self):
        return float(np.linalg.norm(self.centers - self.centers[0], axis=-1).max())

    @property
    def kind(self):
        # View 0 rides on the wrist; views 1 and 2 are bolted to the bench.
        return (f"wrist, moves {self.camera_motion_m:.2f}m" if self.index == 0
                else "fixed exterior")

    @functools.cached_property
    def image_hw(self):
        return self.image(0).shape[:2]


@dataclass(eq=False)
class Episode:
    name: str
    tracks_xyz: np.ndarray        # (T, N, 3) world coordinates -- the 3D ground truth
    queries_xytv: np.ndarray      # (N, 4) x, y, t, view
    views: list

    @property
    def num_frames(self):
        return self.tracks_xyz.shape[0]

    @property
    def num_tracks(self):
        return self.tracks_xyz.shape[1]

    @property
    def num_views(self):
        return len(self.views)

    @functools.cached_property
    def visibility(self):
        return np.stack([view.visibility for view in self.views], axis=-1)  # (T, N, V)

    @functools.lru_cache(maxsize=8)
    def project(self, view):
        """The 2D ground truth for one view: (xy, z), both (T, N, ...)."""
        data = self.views[view]
        return project_tracks(self.tracks_xyz, data.intrinsics, data.extrinsics_w2c)


@functools.lru_cache(maxsize=4)
def load_episode(name):
    """Open one episode. Images stay compressed until a frame is asked for."""
    root = DATA_ROOT / name
    views = []
    for index in sorted(int(p.name) for p in root.iterdir()
                        if p.is_dir() and p.name.isdigit()):
        path = root / str(index)
        views.append(View(
            index=index,
            intrinsics=np.load(path / "intrinsics.npy").astype(np.float32),
            extrinsics_w2c=np.load(path / "extrinsics_w2c.npy").astype(np.float32),
            visibility=np.load(path / "visibility.npy"),
            jpegs=np.load(path / "images_jpeg_bytes.npy", allow_pickle=True),
            path=path))
    return Episode(name=name,
                   tracks_xyz=np.load(root / "tracks_xyz.npy"),
                   queries_xytv=np.load(root / "queries_xytv.npy"),
                   views=views)


In [ ]:
#@title Drawing — the same colours in 2D and in 3D (run once)
def track_colors(track_ids, colormap="turbo"):
    """One colour per drawn track. The 2D dots and the 3D lines share it, so a
    point can be followed from a camera image into the 3D plot and back."""
    fractions = np.linspace(0.05, 0.95, max(len(np.asarray(track_ids)), 1))
    cmap = matplotlib.colormaps[colormap]
    return (np.array([cmap(f)[:3] for f in fractions]) * 255).astype(np.uint8)


def view_color(view):
    """Per-camera colour, used for the camera rig in 3D and the panel borders."""
    palette = np.array([[228, 92, 74], [74, 160, 228], [96, 200, 110], [220, 170, 60]])
    return palette[view % len(palette)]


def pick_tracks(episode, count=24, *, frame=None, require_views=2, seed=7):
    """Sample tracks worth drawing: seen by several cameras, visible right now."""
    visibility = episode.visibility
    eligible = visibility.any(axis=0).sum(axis=-1) >= require_views
    if frame is not None:
        eligible &= visibility[frame].any(axis=-1)
    candidates = np.flatnonzero(eligible)
    if len(candidates) <= count:
        return candidates
    return np.sort(np.random.default_rng(seed).choice(candidates, count, replace=False))


def draw_points(image, xy, *, visible=None, colors=None, radius=4):
    """Draw the 2D ground truth: filled when visible, hollow when occluded."""
    canvas = np.ascontiguousarray(image.copy())
    height, width = canvas.shape[:2]
    colors = track_colors(np.arange(len(xy))) if colors is None else colors
    for index, point in enumerate(xy):
        if not np.isfinite(point).all():
            continue
        x, y = int(round(float(point[0]))), int(round(float(point[1])))
        if not (-radius <= x < width + radius and -radius <= y < height + radius):
            continue
        color = tuple(int(c) for c in colors[index % len(colors)])
        is_visible = True if visible is None else bool(visible[index])
        cv2.circle(canvas, (x, y), radius, color, -1 if is_visible else 1, cv2.LINE_AA)
        if is_visible:
            cv2.circle(canvas, (x, y), radius, (255, 255, 255), 1, cv2.LINE_AA)
    return canvas


def draw_trails(canvas, trail_xy, colors, *, valid=None):
    """Draw per-track trails from (L, N, 2) positions in the current frame."""
    length = trail_xy.shape[0]
    for index in range(trail_xy.shape[1]):
        color = tuple(int(c) for c in colors[index % len(colors)])
        for step in range(1, length):
            start, end = trail_xy[step - 1, index], trail_xy[step, index]
            if not (np.isfinite(start).all() and np.isfinite(end).all()):
                continue
            if valid is not None and not (valid[step - 1, index] and valid[step, index]):
                continue
            cv2.line(canvas,
                     (int(round(float(start[0]))), int(round(float(start[1])))),
                     (int(round(float(end[0]))), int(round(float(end[1])))),
                     color, max(1, int(round(2.0 * step / length))), cv2.LINE_AA)
    return canvas


def ascii_only(text):
    """cv2's Hershey font has no glyphs outside ASCII; anything else draws as "?"."""
    return text.encode("ascii", "replace").decode("ascii")


def label_panel(image, text):
    """Stamp a caption in the top-left corner of a panel."""
    text = ascii_only(text)
    canvas = np.ascontiguousarray(image)
    scale = max(0.5, canvas.shape[1] / 900.0)
    origin = (int(8 * scale), int(28 * scale))
    for color, thickness in (((0, 0, 0), int(4 * scale)),
                             ((255, 255, 255), max(1, int(1.5 * scale)))):
        cv2.putText(canvas, text, origin, cv2.FONT_HERSHEY_SIMPLEX, 0.7 * scale,
                    color, thickness, cv2.LINE_AA)
    return canvas


def letterbox(image, cell_hw, background=(0, 0, 0)):
    """Fit an image into a cell, preserving aspect ratio."""
    cell_h, cell_w = cell_hw
    height, width = image.shape[:2]
    scale = min(cell_w / width, cell_h / height)
    resized = cv2.resize(image, (max(1, int(width * scale)), max(1, int(height * scale))),
                         interpolation=cv2.INTER_AREA)
    canvas = np.full((cell_h, cell_w, 3), np.array(background, dtype=np.uint8), dtype=np.uint8)
    top, left = (cell_h - resized.shape[0]) // 2, (cell_w - resized.shape[1]) // 2
    canvas[top:top + resized.shape[0], left:left + resized.shape[1]] = resized
    return canvas


def montage(panels, *, columns=None, cell_width=480, background=(0, 0, 0)):
    """Tile panels into one grid image."""
    assert panels
    columns = columns or min(len(panels), int(np.ceil(np.sqrt(len(panels)))))
    rows = int(np.ceil(len(panels) / columns))
    aspect = max(panel.shape[0] / panel.shape[1] for panel in panels)
    cell_hw = (int(cell_width * aspect), cell_width)
    cells = [letterbox(panel, cell_hw, background) for panel in panels]
    blank = np.full((*cell_hw, 3), np.array(background, dtype=np.uint8), dtype=np.uint8)
    cells += [blank] * (rows * columns - len(cells))
    return np.vstack([np.hstack(cells[r * columns:(r + 1) * columns]) for r in range(rows)])


def header_panel(image, text, *, bar=26):
    """Put the caption in a bar above the panel instead of over the pixels."""
    text = ascii_only(text)
    canvas = np.ascontiguousarray(image)
    scale = max(0.42, canvas.shape[1] / 1100.0)
    bar = max(bar, int(30 * scale))
    strip = np.full((bar, canvas.shape[1], 3), 22, dtype=np.uint8)
    cv2.putText(strip, text, (int(8 * scale), int(bar * 0.72)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55 * scale, (235, 235, 235),
                max(1, int(1.4 * scale)), cv2.LINE_AA)
    return np.vstack([strip, canvas])


def view_panel(episode, view, frame, track_ids, *, colors=None, trail=0, label=True,
               width=None):
    """One view's frame with the 2D ground truth drawn on it.

    ``width`` renders straight to the size the panel will be shown at, instead of
    drawing on the 1280-wide frame and shrinking afterwards. That matters: a
    4-pixel dot drawn at full resolution survives a 5x downscale as a single
    grey pixel, which is useless for judging whether the point is on target.

    Trails project *past world positions* through the *current* frame's camera,
    so a point that never moves in the world leaves no trail even in the wrist
    view, which is swinging through the scene. Trail length is world motion.
    """
    data = episode.views[view]
    canvas = data.image(frame)
    scale = 1.0
    if width is not None and width != canvas.shape[1]:
        scale = width / canvas.shape[1]
        canvas = cv2.resize(canvas, (width, max(1, int(round(canvas.shape[0] * scale)))),
                            interpolation=cv2.INTER_AREA)
    colors = track_colors(track_ids) if colors is None else colors

    if trail > 1:
        start = max(0, frame - trail + 1)
        trail_xyz = episode.tracks_xyz[start:frame + 1, track_ids]
        extrinsics = np.repeat(data.extrinsics_w2c[frame][None], len(trail_xyz), axis=0)
        trail_xy, trail_z = project_tracks(trail_xyz, data.intrinsics, extrinsics)
        canvas = draw_trails(canvas, trail_xy * scale, colors, valid=trail_z > 1e-3)

    xy, z = episode.project(view)
    visible = data.visibility[frame, track_ids] & (z[frame, track_ids] > 0)
    canvas = draw_points(canvas, xy[frame, track_ids] * scale, visible=visible,
                         colors=colors,
                         radius=max(3, int(round(canvas.shape[1] / 150))))
    if label:
        canvas = header_panel(canvas, f"view {view} ({data.kind})  "
                                      f"frame {frame}/{episode.num_frames - 1}  "
                                      f"vis {int(visible.sum())}/{len(track_ids)}")
    return canvas


def all_views_panel(episode, frame, track_ids, *, colors=None, cell_width=560,
                    trail=0, label=True):
    """The 2D ground truth at one frame, in every view, same colours throughout."""
    colors = track_colors(track_ids) if colors is None else colors
    panels = [view_panel(episode, view, frame, track_ids, colors=colors, trail=trail,
                         label=label, width=cell_width)
              for view in range(episode.num_views)]
    return montage(panels, columns=episode.num_views, cell_width=cell_width)


def episode_video(episode, track_ids, *, colors=None, num_frames=120, cell_width=380,
                  trail=12, label=True):
    """The whole episode with the 2D ground truth burned in, all views."""
    colors = track_colors(track_ids) if colors is None else colors
    frames = np.unique(np.linspace(0, episode.num_frames - 1,
                                   min(num_frames, episode.num_frames)).astype(int))
    return np.stack([all_views_panel(episode, int(frame), track_ids, colors=colors,
                                     cell_width=cell_width, trail=trail, label=label)
                     for frame in frames])


In [ ]:
#@title The 3D ground truth — interactive and static renderers (run once)
# DROID's stereo depth is only trustworthy inside the workspace; the benchmark
# zeroes tracker-input depth past this, so the point cloud stops here too.
MAX_DEPTH_M = 2.0


def unproject_depth(depth, image, intrinsics, extrinsics_w2c, *, stride=6,
                    max_depth_m=MAX_DEPTH_M):
    """Lift one depth frame into a world point cloud; returns (points, colors)."""
    height, width = depth.shape
    rows, columns = np.mgrid[0:height:stride, 0:width:stride]
    z = depth[::stride, ::stride]
    valid = np.isfinite(z) & (z > 0.0)
    if max_depth_m is not None:
        valid &= z <= max_depth_m
    fx, fy, cx, cy = [float(value) for value in intrinsics]
    points_camera = np.stack([(columns - cx) / fx * z, (rows - cy) / fy * z, z], axis=-1)[valid]
    rotation, translation = extrinsics_w2c[:3, :3], extrinsics_w2c[:3, 3]
    points_world = (points_camera - translation) @ rotation
    scale_h, scale_w = image.shape[0] / height, image.shape[1] / width
    color_rows = np.clip((rows * scale_h).astype(np.int64), 0, image.shape[0] - 1)
    color_columns = np.clip((columns * scale_w).astype(np.int64), 0, image.shape[1] - 1)
    return points_world.astype(np.float32), image[color_rows, color_columns][valid]


def hex_colors(colors):
    """Vectorized #rrggbb strings for plotly, one per point."""
    colors = np.asarray(colors, dtype=np.uint8).reshape(-1, 3)
    packed = (colors[:, 0].astype(np.uint32) << 16 | colors[:, 1].astype(np.uint32) << 8
              | colors[:, 2].astype(np.uint32))
    return ["#%06x" % value for value in packed]


def plot_tracks_3d(episode, track_ids, *, frame=None, colors=None, cloud_views=(1, 2),
                   cloud_stride=6, max_cloud_points=40_000, show_cloud=True,
                   title=None, height=760):
    """The 3D ground truth, rotatable: whole trajectories, heads, and the rig.

    The depth cloud is the reference surface. If the 3D tracks are right they sit
    *on* it -- points floating off the cloud, or sunk into it, are the thing to
    look for.
    """
    import plotly.graph_objects as go

    frame = episode.num_frames // 2 if frame is None else frame
    track_ids = np.asarray(track_ids)
    colors = track_colors(track_ids) if colors is None else np.asarray(colors)
    figure = go.Figure()

    if show_cloud:
        for view in cloud_views:
            depth = episode.views[view].depth(frame)
            if depth is None:
                continue
            points, point_colors = unproject_depth(
                depth, episode.views[view].image(frame), episode.views[view].intrinsics,
                episode.views[view].extrinsics_w2c[frame], stride=cloud_stride)
            if not len(points):
                continue
            if len(points) > max_cloud_points:
                keep = np.random.default_rng(7).choice(len(points), max_cloud_points,
                                                       replace=False)
                points, point_colors = points[keep], point_colors[keep]
            figure.add_trace(go.Scatter3d(
                x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
                marker=dict(size=1.5, color=hex_colors(point_colors)),
                name=f"depth cloud, view {view}", hoverinfo="skip"))

    # Whole trajectories, one NaN-separated trace: far fewer traces to serialize.
    paths = episode.tracks_xyz[:, track_ids]                          # (T, K, 3)
    separator = np.full((1, len(track_ids), 3), np.nan, dtype=np.float32)
    joined = np.concatenate([paths, separator]).transpose(1, 0, 2).reshape(-1, 3)
    figure.add_trace(go.Scatter3d(
        x=joined[:, 0], y=joined[:, 1], z=joined[:, 2], mode="lines",
        line=dict(color=np.repeat(hex_colors(colors), paths.shape[0] + 1).tolist(),
                  width=4),
        name="3D tracks (whole episode)", hoverinfo="skip"))

    heads = episode.tracks_xyz[frame, track_ids]
    figure.add_trace(go.Scatter3d(
        x=heads[:, 0], y=heads[:, 1], z=heads[:, 2], mode="markers",
        marker=dict(color=hex_colors(colors), size=5),
        text=[f"track {int(track)}" for track in track_ids], hoverinfo="text",
        name=f"positions at frame {frame}"))

    for view, data in enumerate(episode.views):
        centers = data.centers
        color = "rgb({},{},{})".format(*view_color(view))
        figure.add_trace(go.Scatter3d(
            x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode="lines",
            line=dict(width=4, color=color), name=f"camera {view} path",
            hoverinfo="skip"))
        figure.add_trace(go.Scatter3d(
            x=[centers[frame, 0]], y=[centers[frame, 1]], z=[centers[frame, 2]],
            mode="markers+text", marker=dict(size=5, symbol="diamond", color=color),
            text=[f"v{view}"], textposition="top center", name=f"camera {view}"))

    figure.update_layout(
        height=height, margin=dict(l=0, r=0, t=34, b=0),
        title=title if title is not None else f"{episode.name} — 3D ground truth",
        scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"))
    return figure


def draw_tracks_3d_axes(ax, episode, track_ids, *, colors=None, elev=22, azim=-60,
                        linewidth=0.9, show_cameras=True):
    """The same 3D ground truth as a small static panel, for contact sheets."""
    track_ids = np.asarray(track_ids)
    colors = (track_colors(track_ids) if colors is None else np.asarray(colors)) / 255.0
    paths = episode.tracks_xyz[:, track_ids]                          # (T, K, 3)
    for index in range(paths.shape[1]):
        ax.plot(*paths[:, index].T, color=colors[index % len(colors)], lw=linewidth)
    ax.scatter(*paths[-1].T, color=colors[: paths.shape[1]], s=4, depthshade=False)
    if show_cameras:
        for view, data in enumerate(episode.views):
            centers = data.centers
            color = view_color(view) / 255.0
            if data.camera_motion_m < 0.01:
                ax.scatter(*centers[0], color=color, s=26, marker="D", depthshade=False)
            else:
                ax.plot(*centers.T, color=color, lw=1.6)
    # Frame the box on the tracks. The cameras can sit a metre outside the
    # workspace, and letting them set the limits shrinks the tracks to a smudge.
    flat = paths.reshape(-1, 3)
    low, high = flat.min(axis=0), flat.max(axis=0)
    span = np.maximum(high - low, 1e-3)
    low, high = low - 0.15 * span, high + 0.15 * span
    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[1], high[1])
    ax.set_zlim(low[2], high[2])
    ax.set_box_aspect(high - low)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    return ax


In [ ]:
EPISODES = find_episodes()
print(f"{len(EPISODES)} episodes under {DATA_ROOT}")
print("\n".join(f"  [{index:2d}] {name}" for index, name in enumerate(EPISODES[:5])))
print("   ...")


## 2. Pick one episode to look at closely


In [ ]:
EPISODE_INDEX = 0   #@param {type:"slider", min:0, max:49, step:1}
NUM_TRACKS = 24     # how many tracks to draw
TRAIL = 12          # trail length, in frames
FRAME = None        # None = the midpoint

episode = load_episode(EPISODES[EPISODE_INDEX])
FRAME = episode.num_frames // 2 if FRAME is None else FRAME
track_ids = pick_tracks(episode, NUM_TRACKS, frame=FRAME, require_views=2)
colors = track_colors(track_ids)          # shared by every 2D and 3D panel below

print(f"{episode.name}  [{EPISODE_INDEX}]")
print(f"  {episode.num_frames} frames x {episode.num_tracks} tracks x {episode.num_views} views"
      f"  |  images {episode.views[0].image_hw}")
for view, data in enumerate(episode.views):
    xy, z = episode.project(view)
    print(f"  view {view}: {data.kind:22s} visible {data.visibility.mean():.3f}"
          f"  median track depth {np.median(z[data.visibility]):.2f} m")
print(f"  drawing tracks {track_ids[:8].tolist()}{' ...' if len(track_ids) > 8 else ''}"
      f" at frame {FRAME}")


## 3. The 2D track ground truth

### 3.1 One frame, all three views

The same 3D points, the same colours, in every camera at the same instant. This is the panel that
catches a wrong 3D position fastest: a point can look plausible in one view and be obviously off
the object in another.


In [ ]:
panel = all_views_panel(episode, FRAME, track_ids, colors=colors, cell_width=640, trail=TRAIL)
plt.figure(figsize=(17, 17 * panel.shape[0] / panel.shape[1]))
plt.imshow(panel)
plt.axis("off")
plt.title(f"{episode.name} — 2D ground truth, frame {FRAME}")
plt.show()


### 3.2 The whole episode

Watch whether each dot stays glued to its physical point. A dot that slides across a surface, or
that stays filled while the gripper passes in front of it, is a ground-truth error.


In [ ]:
video = episode_video(episode, track_ids, colors=colors, num_frames=120,
                      cell_width=440, trail=TRAIL)
mediapy.show_video(video, fps=12, title=f"{episode.name} — 2D tracks, all views")


### 3.3 One track at a time

Pick a single track and watch it alone in all three views. With 24 dots on screen it is hard to
tell which is which; with one, an error is unmissable. The crop follows the point.


In [ ]:
FOCUS = int(track_ids[int(np.argmax(episode.visibility[FRAME][track_ids].sum(axis=-1)))])
PATCH = 180        # crop size in source pixels

def focus_strip(episode, track, frame, *, patch=PATCH, out_size=260):
    """The same track cropped out of every view, with a crosshair on the GT pixel."""
    panels = []
    for view in range(episode.num_views):
        xy, z = episode.project(view)
        point = xy[frame, track]
        image = episode.views[view].image(frame)
        height, width = image.shape[:2]
        half = patch // 2
        x = int(round(float(np.clip(point[0], half, width - half - 1))))
        y = int(round(float(np.clip(point[1], half, height - half - 1))))
        crop = cv2.resize(image[y - half:y + half, x - half:x + half],
                          (out_size, out_size), interpolation=cv2.INTER_NEAREST)
        scale = out_size / patch
        center = (int(round((float(point[0]) - (x - half)) * scale)),
                  int(round((float(point[1]) - (y - half)) * scale)))
        visible = bool(episode.views[view].visibility[frame, track]) and float(z[frame, track]) > 0
        color = (60, 220, 60) if visible else (220, 60, 60)
        cv2.drawMarker(crop, center, color, cv2.MARKER_CROSS, int(out_size * 0.22), 2,
                       cv2.LINE_AA)
        cv2.rectangle(crop, (0, 0), (out_size - 1, out_size - 1), color, 3)
        panels.append(label_panel(crop, f"v{view} {'vis' if visible else 'occl'}"
                                        f" z={float(z[frame, track]):.2f}"))
    return montage(panels, columns=episode.num_views, cell_width=out_size)

frames = np.unique(np.linspace(0, episode.num_frames - 1, 60).astype(int))
focus_video = np.stack([focus_strip(episode, FOCUS, int(frame)) for frame in frames])
mediapy.show_video(focus_video, fps=10,
                   title=f"track {FOCUS} alone — green border = visible, red = occluded")


## 4. The 3D track ground truth

The depth cloud from the two fixed cameras is the reference surface. Correct 3D tracks lie **on**
it: a curve floating in mid-air or buried inside the table is wrong. Drag to rotate.

The camera rig is drawn too — two diamonds for the fixed exteriors and an arc for the wrist camera,
which is the arm's own path.


In [ ]:
figure = plot_tracks_3d(episode, track_ids, frame=FRAME, colors=colors,
                        cloud_views=(1, 2), show_cloud=True,
                        title=f"{episode.name} — 3D ground truth, cloud at frame {FRAME}")
figure.show()


### 4.1 3D and 2D side by side

The same instant and the same colours in both rows: the three camera images on top, the 3D tracks
below from three viewpoints. Matching a 3D curve to its 2D dots is the whole verification, and one
viewpoint is rarely enough to tell whether a curve really sits on the surface.


In [ ]:
AZIMUTHS = (-60, 10, 80)     # three viewpoints on the same 3D tracks

figure_3d = plt.figure(figsize=(16, 8.4))
for column, view in enumerate(range(episode.num_views)):
    ax = figure_3d.add_subplot(2, 3, column + 1)
    ax.imshow(view_panel(episode, view, FRAME, track_ids, colors=colors, trail=TRAIL,
                         label=False, width=640))
    ax.set_title(f"view {view} — {episode.views[view].kind}", fontsize=9)
    ax.axis("off")
for column, azimuth in enumerate(AZIMUTHS):
    ax = figure_3d.add_subplot(2, 3, column + 4, projection="3d")
    draw_tracks_3d_axes(ax, episode, track_ids, colors=colors, azim=azimuth)
    ax.set_title(f"3D tracks, azimuth {azimuth}°", fontsize=9)
figure_3d.suptitle(f"{episode.name} — frame {FRAME}  (same colours in both rows)",
                   fontsize=11)
figure_3d.tight_layout()


## 5. Every episode — the 2D tracks

The sweep. Both cells below run over all 50 episodes; drop `SWEEP_EPISODES` to a slice while you
are still adjusting the knobs.


In [ ]:
SWEEP_EPISODES = EPISODES          # e.g. EPISODES[:8] for a quick pass
SWEEP_TRACKS = 20                  # tracks drawn per episode
SWEEP_FRAMES = 40                  # frames per episode video
SWEEP_CELL = 260                   # pixels per view
SWEEP_TRAIL = 10

def sweep_tracks(episode, count=SWEEP_TRACKS):
    """The same deterministic choice of tracks for every episode."""
    return pick_tracks(episode, count, frame=episode.num_frames // 2, require_views=2)

print(f"{len(SWEEP_EPISODES)} episodes x {SWEEP_TRACKS} tracks x 3 views")


### 5.1 One frame per episode

All 50 at once: three views each, ground truth drawn. Anything that looks wrong here is worth
opening in section 2 by its index.


In [ ]:
tiles = []
for index, name in enumerate(SWEEP_EPISODES):
    other = load_episode(name)
    frame = other.num_frames // 2
    ids = sweep_tracks(other)
    tile = all_views_panel(other, frame, ids, colors=track_colors(ids), cell_width=300,
                           trail=SWEEP_TRAIL, label=False)
    tiles.append(label_panel(tile, f"[{index}] {name}"))

sheet = montage(tiles, columns=2, cell_width=900, background=(20, 20, 20))
plt.figure(figsize=(16, 16 * sheet.shape[0] / sheet.shape[1]))
plt.imshow(sheet)
plt.axis("off")
plt.title(f"2D ground truth — {len(SWEEP_EPISODES)} episodes")
plt.show()


### 5.2 Every episode as a video

One player per episode, three views in each, ground truth burned in. This is the "watch all the
ground truth" cell.

Fifty clips is a few minutes of decoding and roughly 1 GB of RAM at these settings; the notebook
also gets large once they are all embedded, so clear the outputs before committing it.


In [ ]:
videos = {}
for index, name in enumerate(SWEEP_EPISODES):
    other = load_episode(name)
    ids = sweep_tracks(other)
    videos[f"[{index}] {name.split('+')[0]}"] = episode_video(
        other, ids, colors=track_colors(ids), num_frames=SWEEP_FRAMES,
        cell_width=SWEEP_CELL, trail=SWEEP_TRAIL, label=False)

print(f"{len(videos)} clips, {sum(v.nbytes for v in videos.values()) / 1e9:.2f} GB in RAM")
mediapy.show_videos(videos, fps=12, columns=2, height=190)


## 6. Every episode — the 3D tracks

### 6.1 All 50 trajectories on one sheet

Each panel is one episode's complete set of 3D tracks, plus its camera rig (diamonds = the fixed
exteriors, arc = the wrist camera). What you are looking for is shape: a tabletop workspace with a
compact cluster of curves. A panel with curves shooting off to infinity, or a cloud with no
structure, is an episode to open individually.


In [ ]:
SHEET_TRACKS = 80        # tracks per panel
COLUMNS = 5

rows = int(np.ceil(len(SWEEP_EPISODES) / COLUMNS))
figure = plt.figure(figsize=(3.1 * COLUMNS, 3.1 * rows))
for index, name in enumerate(SWEEP_EPISODES):
    other = load_episode(name)
    ids = pick_tracks(other, SHEET_TRACKS, require_views=1)
    ax = figure.add_subplot(rows, COLUMNS, index + 1, projection="3d")
    draw_tracks_3d_axes(ax, other, ids, colors=track_colors(ids), linewidth=0.7)
    ax.set_title(f"[{index}] {name.split('+')[0]}", fontsize=7)
figure.suptitle("3D ground-truth tracks — every episode", fontsize=12)
figure.tight_layout()
plt.show()


### 6.2 Any episode, interactive

The sheet above finds the suspicious ones; this opens one. Set the index and rotate it, with the
depth cloud on so you can see whether the curves sit on the surfaces.


In [ ]:
INSPECT_INDEX = EPISODE_INDEX     #@param {type:"integer"}

other = load_episode(EPISODES[INSPECT_INDEX])
ids = pick_tracks(other, 60, frame=other.num_frames // 2, require_views=1)
plot_tracks_3d(other, ids, frame=other.num_frames // 2, colors=track_colors(ids),
               show_cloud=True,
               title=f"[{INSPECT_INDEX}] {other.name} — 3D ground truth").show()


## 7. Save

Per-episode MP4s of the 2D ground truth and standalone rotatable HTML of the 3D ground truth, so
they can be reviewed outside the notebook or sent to someone else.


In [ ]:
SAVE = False      #@param {type:"boolean"}

out_dir = Path("droid_track_gt")
if SAVE:
    import imageio.v3 as iio

    out_dir.mkdir(exist_ok=True)
    for index, name in enumerate(SWEEP_EPISODES):
        other = load_episode(name)
        ids = sweep_tracks(other)
        stem = f"{index:02d}_{name.replace('+', '_')}"
        mediapy.write_video(out_dir / f"{stem}_2d.mp4",
                            episode_video(other, ids, colors=track_colors(ids),
                                          num_frames=SWEEP_FRAMES, cell_width=440,
                                          trail=SWEEP_TRAIL),
                            fps=12)
        plot_tracks_3d(other, ids, frame=other.num_frames // 2,
                       colors=track_colors(ids), show_cloud=True).write_html(
            out_dir / f"{stem}_3d.html", include_plotlyjs="cdn")
        print(f"  {stem}")
    print(f"wrote {len(SWEEP_EPISODES)} pairs to {out_dir.resolve()}")
else:
    print("SAVE = False — flip it to write the MP4s and the 3D HTML files")
